# Star Graph Pipeline on Kaggle

**Boots a local llama.cpp LLM via harness.py, then runs your star-graph
enrichment pipeline using the local model instead of NVIDIA NIM. Pushes
results back to GitHub.**

**Prerequisites:**
- GPU: 1x T4 (or P100)
- Secrets: `GH_PAT` (Kaggle Secret with `repo` scope)
- First boot: downloads GGUF (~2.5GB) + builds llama.cpp (~15 min)
- Cached boot: ~2 min, pipeline ~10 min

**Other models:** Change MODEL_KEY below to any model in the kaggle-model-server
registry — this notebook doesn't contain model logic, just the pipeline.

In [ ]:
# --- clone both repos ---------------------------------------------------
!rm -rf /kaggle/working/kms /kaggle/working/star-graph
!git clone https://github.com/Meru143/kaggle-model-server.git /kaggle/working/kms
!git clone https://github.com/Meru143/star-graph.git /kaggle/working/star-graph

import sys
sys.path.insert(0, "/kaggle/working/kms")
sys.path.insert(0, "/kaggle/working/star-graph/kaggle")

!pip install -q huggingface_hub requests networkx numpy sentence-transformers

In [ ]:
# --- fresh imports ------------------------------------------------------
import importlib, os, sys
for _m in ("model_registry", "harness"):
    if _m in sys.modules:
        if _m == "harness":
            try: sys.modules["harness"].stop()
            except Exception: pass
        importlib.reload(sys.modules[_m])

from model_registry import MODELS
from harness import run, stop, list_quants, harvest_cache
from star_graph_kaggle import run_pipeline

# expose GH_PAT so the pipeline can push to GitHub
import kaggle_secrets
try:
    os.environ.setdefault("GH_PAT", kaggle_secrets.UserSecretsClient().get_secret("GH_PAT"))
except Exception:
    print("WARNING: GH_PAT secret not found — can't push results. Set it in Kaggle Secrets.")

list(MODELS.keys())

In [ ]:
# --- boot the LLM -------------------------------------------------------
# Pick any model from the registry. Nanbeige4.2-3B is small, fast, JSON-safe.
# Gemma4-12B gives better analysis quality if you have time.

MODEL_KEY = "owao/Nanbeige4.2-3B-GGUF"

url = run(MODEL_KEY, MODELS, quant="Q4_K_M", ctx=4096)
print(f"Model running at {url}")

In [ ]:
# --- run ----------------------------------------------------------------
# limit=None = all repos. Start with limit=5 to test.

run_pipeline(limit=None)

In [ ]:
# --- harvest cache (do before ending session) ---------------------------
harvest_cache()

## Cleanup

Run `stop()` then **end the Kaggle session** to stop GPU burn.